# In-Class Exercise: Python Basics & Your First OLS

**Course:** ECON6083 - Machine Learning in Economics
**Topic:** Python Fundamentals and Supervised Learning with OLS
**Time:** 25 minutes

---

## Objective

This exercise introduces the Python tools you will use throughout the course. You will learn to: (1) work with NumPy arrays, (2) define functions, (3) run OLS regression using `statsmodels`, and (4) use `scikit-learn` for prediction.

**Setup:** Run the following in a Jupyter notebook or Python script.

In [3]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
from sklearn.linear_model import LinearRegression

---

## Part I: Python & NumPy Basics (5 minutes)

### 1a. Arrays and Vectorized Operations

In [4]:
# Create arrays
x = np.array([1, 2, 3, 4, 5])
y = np.array([2.1, 3.9, 6.2, 7.8, 10.1])

# Vectorized operations (no loops needed!)
print("Mean of x:", np.mean(x))
print("x squared:", x ** 2)
print("x * y:", x * y)

Mean of x: 3.0
x squared: [ 1  4  9 16 25]
x * y: [ 2.1  7.8 18.6 31.2 50.5]


**Try it:** compute the sample covariance of `x` and `y` manually:
$$\text{Cov}(x, y) = \frac{1}{n-1} \sum_{i=1}^{n} (x_i - \bar{x})(y_i - \bar{y})$$

In [5]:
# TODO: Compute the sample covariance
n = len(x)
cov_xy = sum(x*y-x*np.mean(y)-y*np.mean(x)+np.mean(x)*np.mean(y))/(n-1)  # hint: use np.sum(), np.mean()
print(f"Manual covariance: {cov_xy:.4f}")

# Verify with NumPy
print(f"NumPy covariance:  {np.cov(x, y)[0, 1]:.4f}")

Manual covariance: 4.9750
NumPy covariance:  4.9750


### 1b. Random Number Generation

In [6]:
# Set seed for reproducibility (you will see this in every assignment!)
np.random.seed(42)

# Draw 1000 samples from N(0, 1)
z = np.random.normal(0, 1, 1000)
print(f"Sample mean: {np.mean(z):.3f} (should be close to 0)")
print(f"Sample std:  {np.std(z, ddof=1):.3f} (should be close to 1)")

Sample mean: 0.019 (should be close to 0)
Sample std:  0.979 (should be close to 1)


---

## Part II: Defining Functions (5 minutes)

In this course, you will frequently write functions to encapsulate estimation procedures.

### 2a. A Simple Function

In [7]:
def ols_beta_hat(X, y):
    """
    Compute the OLS estimator using the normal equation:
        beta_hat = (X'X)^{-1} X'y

    Parameters
    ----------
    X : np.ndarray, shape (n, k) — design matrix (include constant if needed)
    y : np.ndarray, shape (n,)   — outcome vector

    Returns
    -------
    beta_hat : np.ndarray, shape (k,)
    """
    # TODO: Implement the OLS normal equation
    # hint: np.linalg.inv() for inverse, @ for matrix multiplication, .T for transpose
    beta_hat = np.linalg.inv(X.T @ X) @ X.T @ y
    return beta_hat

### 2b. Test Your Function

In [8]:
# Generate data: y = 3 + 2*x + noise
np.random.seed(42)
n = 100
x = np.random.normal(0, 1, n)
y = 3 + 2 * x + np.random.normal(0, 1, n)

# Add constant column to X
X = np.column_stack([np.ones(n), x])  # shape (100, 2)

# Run your function
beta = ols_beta_hat(X, y)
print(f"Intercept: {beta[0]:.3f} (true = 3)")
print(f"Slope:     {beta[1]:.3f} (true = 2)")

Intercept: 3.007 (true = 3)
Slope:     1.857 (true = 2)


**Question:** Why do we need to manually add a column of ones to `X`?

> Your answer: To include and estimate the intercept term (constant term) in the linear regression model.

---

## Part III: OLS with `statsmodels` (7 minutes)

`statsmodels` is the standard Python package for econometric inference — it gives you coefficients, standard errors, t-statistics, and confidence intervals.

### 3a. Basic OLS

In [9]:
# Using the same x, y from Part II
X_sm = sm.add_constant(x)  # adds the constant for you
model = sm.OLS(y, X_sm).fit()
print(model.summary())

                            OLS Regression Results                            
Dep. Variable:                      y   R-squared:                       0.761
Model:                            OLS   Adj. R-squared:                  0.759
Method:                 Least Squares   F-statistic:                     312.2
Date:                Wed, 18 Mar 2026   Prob (F-statistic):           3.14e-32
Time:                        14:46:45   Log-Likelihood:                -135.71
No. Observations:                 100   AIC:                             275.4
Df Residuals:                      98   BIC:                             280.6
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const          3.0074      0.096     31.465      0.0

**Questions:**
1. Find the R-squared in the output. What does it mean?
2. Are the coefficients statistically significant at the 5% level? How do you tell?
3. Compare the coefficients to your `ols_beta_hat()` results. Do they match?

> Your answers:
> 1. 0.761
> 2. Yes, as we can see p-values from the table, which are both 0.000, far below the 5% threshold. A p-value less than 0.05 means we reject the null hypothesis that the true population coefficient equals 0, and conclude the coefficient is statistically significant.
> 3. Yes. They are exactly the same.

### 3b. Multiple Regression

Now let's add a second regressor and see what happens.

In [10]:
# Generate a second feature that is correlated with x
np.random.seed(42)
x1 = np.random.normal(0, 1, n)
x2 = 0.5 * x1 + np.random.normal(0, 1, n)  # correlated with x1
y = 3 + 2 * x1 + 0 * x2 + np.random.normal(0, 1, n)  # x2 has NO true effect

X_mult = sm.add_constant(np.column_stack([x1, x2]))
model_mult = sm.OLS(y, X_mult).fit()
print(model_mult.summary())

                            OLS Regression Results                            
Dep. Variable:                      y   R-squared:                       0.783
Model:                            OLS   Adj. R-squared:                  0.779
Method:                 Least Squares   F-statistic:                     175.3
Date:                Wed, 18 Mar 2026   Prob (F-statistic):           6.19e-33
Time:                        14:52:23   Log-Likelihood:                -147.62
No. Observations:                 100   AIC:                             301.2
Df Residuals:                      97   BIC:                             309.1
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const          3.0886      0.108     28.537      0.0

**Questions:**
1. What is the coefficient on `x2`? Is it close to the true value (0)?
2. Is the coefficient on `x2` statistically significant?
3. What happened to the standard error of `x1`'s coefficient compared to the simple regression? Why?

> Your answers:
> 1. -0.0123. Yes, the p-value is 0.915, which means that we cannot reject it from 0 at 5% significance level.
> 2. No, the coefficient on x2 is not statistically significant at the standard 5% significance level since the p-value is 0.915>0.05 and the confidence interval contains 0.
> 3. Compared to the simple (univariate) regression, the standard error of x1's coefficient increased. Because there exists imperfect multicollinearity between x1 and x2. When independent variables are correlated, OLS cannot perfectly separate their independent effects on the outcome y, which inflates the variance of the coefficient estimates.

---

## Part IV: Prediction with `scikit-learn` (8 minutes)

`scikit-learn` is the standard ML library. Unlike `statsmodels`, it focuses on **prediction** rather than inference — no p-values, but great for out-of-sample forecasting.

### 4a. Fit and Predict

In [11]:
# Scenario: Predicting CEO salaries from firm characteristics
data = {
    'salary': [1095, 1001, 1122, 578, 1368, 1145, 1078, 1094, 1237, 833],
    'sales':  [27595, 9958, 6121, 3724, 21653, 10706, 26002, 7010, 16596, 6262],
    'mktval': [41344, 14515, 10041, 5173, 28999, 14223, 30839, 10614, 21943, 8004]
}
df = pd.DataFrame(data)

# TODO: Define features and outcome
X = df[['sales', 'mktval']] # hint: df[['sales', 'mktval']]
y = df['salary']  # hint: df['salary']

# TODO: Initialize and fit
model_sk = LinearRegression()  # hint: LinearRegression()
model_sk.fit(X, y)

print(f"Intercept:    {model_sk.intercept_:.2f}")
print(f"Coefficients: {model_sk.coef_}")

Intercept:    866.12
Coefficients: [0.00328586 0.00777718]


### 4b. Make a Prediction

In [12]:
# Predict salary for a firm with sales = $15,000M, mktval = $20,000M
new_firm = pd.DataFrame({'sales': [15000], 'mktval': [20000]})

# TODO: Predict
predicted = model_sk.predict(new_firm)  # hint: model_sk.predict(new_firm)
print(f"Predicted salary: ${predicted[0]:.2f}K")

Predicted salary: $1070.95K


### 4c. Compare `statsmodels` vs `scikit-learn`

In [13]:
# Run the same regression in statsmodels
X_sm = sm.add_constant(df[['sales', 'mktval']])
model_sm = sm.OLS(df['salary'], X_sm).fit()
print(model_sm.summary())

                            OLS Regression Results                            
Dep. Variable:                 salary   R-squared:                       0.305
Model:                            OLS   Adj. R-squared:                  0.106
Method:                 Least Squares   F-statistic:                     1.535
Date:                Wed, 18 Mar 2026   Prob (F-statistic):              0.280
Time:                        15:02:45   Log-Likelihood:                -65.682
No. Observations:                  10   AIC:                             137.4
Df Residuals:                       7   BIC:                             138.3
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const        866.1166    126.220      6.862      0.0

**Question:** The coefficients should be identical. What does `scikit-learn` give you that `statsmodels` doesn't, and vice versa?

> Your answer: scikit-learn gives a more straight predicted value, while statsmodels gives more about the significant degree(p-value, etc) and the degree of model matching(R-squared).

---

## Summary

| Tool | Use Case | Gives You |
|------|----------|-----------|
| **NumPy** | Array operations, linear algebra | Fast vectorized computation |
| **Custom functions** | Reusable estimation procedures | `ols_beta_hat(X, y)` |
| **`statsmodels`** | Econometric inference | Coefficients, SEs, p-values, CIs |
| **`scikit-learn`** | ML prediction | `.fit()`, `.predict()`, cross-validation |

**Key takeaway:** Throughout this course, you will use `statsmodels` when you need **inference** (hypothesis tests, confidence intervals) and `scikit-learn` when you need **prediction** (out-of-sample accuracy, cross-validation).